In [3]:
import uproot
import matplotlib.pyplot as plt
import pandas as pd
import tqdm
import numpy as np
import os
import glob
import awkward as ak
import boost_histogram as bh
import mplhep as mh
from matplotlib.colors import LogNorm
import math
mh.style.use("ATLAS")
import particle
# !pip install pyarrow
# !pip install "pyarrow<21" --break-system-packages

In [53]:
data = uproot.open("build/merged_nominal.root:cutflow")
# data = uproot.open("build/dimuon_selected/merged_nominal.root:cutflow")

In [54]:
df = pd.DataFrame(data.arrays(library="pd"))

In [55]:
display(df)

,cut_name,all,passed,efficiency
0,Remove CaloNu PMT region (truth),541884,541884,100.000000
1,isDimuonn,541884,77,0.014210
2,At least one long track,77,77,100.000000
3,Veto20 and Veto21 charge > 40 pC,77,76,98.701302
4,Timing Station Charge > 20 pC,76,76,100.000000
...,...,...,...,...
130,Track R at IFT < 95 mm,29,24,82.758621
131,Track rVetoNu < 120 mm,24,23,95.833328
132,Leading track theta < 25 mrad,23,22,95.652176
133,VetoNu0 reduced charge < 30 pC,22,18,81.818184


In [56]:
# cutflow tables from each run are appended together in the root file, so we need to merge all entries with the same cut_name. We can do this by grouping by cut_name and summing the other columns.
df = df.groupby("cut_name").sum().reset_index()
df = df.sort_values("all", ascending=False)

# Need to recompute the efficiency column, as it is not meaningful after summing the other columns.
df["efficiency"] = df["passed"] / df["all"]
display(df)

,cut_name,all,passed,efficiency
5,Remove CaloNu PMT region (truth),4209098,4206490,0.999380
14,isDimuonn,4206490,771,0.000183
0,At least one long track,771,771,1.000000
11,Veto20 and Veto21 charge > 40 pC,771,768,0.996109
4,Preshower Charge > 2.5 pC,768,758,0.986979
6,Timing Station Charge > 20 pC,768,768,1.000000
9,Track pz > 50 GeV,758,751,0.990765
1,Leading track has at >= 7 layers,751,751,1.000000
8,Track nDoF >= 9,751,742,0.988016
7,Track R at IFT < 95 mm,742,569,0.766846


In [ ]:
n_dimuon_mc = 771
n_dimuon_asimov = n_dimuon_mc * (186 / 10_000)
print(f"Number of dimuon events in MC: {n_dimuon_mc}")
print(f"Number of dimuon events in Asimov: {n_dimuon_asimov:.2f}")

Number of dimuon events in MC: 771
Number of dimuon events in Asimov: 14.34


In [58]:
n_dimuon_mc = 416
n_dimuon_asimov = n_dimuon_mc * (186 / 10_000)
print(f"Number of dimuon events in MC: {n_dimuon_mc}")
print(f"Number of dimuon events in Asimov: {n_dimuon_asimov:.2f}")

Number of dimuon events in MC: 416
Number of dimuon events in Asimov: 7.74


In [59]:
n_dimuon_mc = 312
n_dimuon_asimov = n_dimuon_mc * (186 / 10_000)
print(f"Number of dimuon events in MC: {n_dimuon_mc}")
print(f"Number of dimuon events in Asimov: {n_dimuon_asimov:.2f}")

Number of dimuon events in MC: 312
Number of dimuon events in Asimov: 5.80


In [7]:
def print_cutflow(filpaths, lumi_sample=10_000, lumi_target=None):
    if isinstance(filpaths, str):
        filpaths = [filpaths]

    df = uproot.concatenate([f"{filpath}:cutflow" for filpath in filpaths], library="pd")
    # cutflow tables from each run are appended together in the root file, so we need to merge all entries with the same cut_name. We can do this by grouping by cut_name and summing the other columns.
    df = df.groupby("cut_name").sum().reset_index()
    df = df.sort_values("all", ascending=False)

    # Need to recompute the efficiency column, as it is not meaningful after summing the other columns.
    df["efficiency"] = df["passed"] / df["all"]

    if lumi_target is not None:
        df["all"] = df["all"] * (lumi_target / lumi_sample)
        df["passed"] = df["passed"] * (lumi_target / lumi_sample)

    display(df)
    
    

In [12]:
print_cutflow("/home/ppd/bewilson/work/new_nuAnalysis/nuAnalysis/build/dimuon_selected/2022_2023.bck.root")
print_cutflow("/home/ppd/bewilson/work/new_nuAnalysis/nuAnalysis/build/dimuon_selected/2022_2023.bck.root", lumi_target=138)

,cut_name,all,passed,efficiency
5,Remove CaloNu PMT region (truth),1493362,1493362,1.000000
15,isDimuonn,1493362,316,0.000212
12,Veto20 and Veto21 charge > 40 pC,316,315,0.996835
0,At least one long track,316,316,1.000000
4,Preshower Charge > 2.5 pC,315,312,0.990476
6,Timing Station Charge > 20 pC,315,315,1.000000
10,Track pz > 50 GeV,312,309,0.990385
1,Leading track has at >= 7 layers,309,309,1.000000
9,Track nDoF >= 9,309,303,0.980583
8,Track R at max radius < 95 mm,303,180,0.594059


,cut_name,all,passed,efficiency
5,Remove CaloNu PMT region (truth),20608.3956,20608.3956,1.000000
15,isDimuonn,20608.3956,4.3608,0.000212
12,Veto20 and Veto21 charge > 40 pC,4.3608,4.3470,0.996835
0,At least one long track,4.3608,4.3608,1.000000
4,Preshower Charge > 2.5 pC,4.3470,4.3056,0.990476
6,Timing Station Charge > 20 pC,4.3470,4.3470,1.000000
10,Track pz > 50 GeV,4.3056,4.2642,0.990385
1,Leading track has at >= 7 layers,4.2642,4.2642,1.000000
9,Track nDoF >= 9,4.2642,4.1814,0.980583
8,Track R at max radius < 95 mm,4.1814,2.4840,0.594059


In [13]:
print_cutflow("/home/ppd/bewilson/work/new_nuAnalysis/nuAnalysis/build/dimuon_selected/2024_noCaloNu.bck.root")
print_cutflow("/home/ppd/bewilson/work/new_nuAnalysis/nuAnalysis/build/dimuon_selected/2024_noCaloNu.bck.root", lumi_target=50)

,cut_name,all,passed,efficiency
5,Remove CaloNu PMT region (truth),1450841,1450841,1.000000
15,isDimuonn,1450841,296,0.000204
12,Veto20 and Veto21 charge > 40 pC,296,294,0.993243
0,At least one long track,296,296,1.000000
4,Preshower Charge > 2.5 pC,294,289,0.982993
6,Timing Station Charge > 20 pC,294,294,1.000000
10,Track pz > 50 GeV,289,286,0.989619
1,Leading track has at >= 7 layers,286,286,1.000000
9,Track nDoF >= 9,286,284,0.993007
8,Track R at max radius < 95 mm,284,170,0.598592


,cut_name,all,passed,efficiency
5,Remove CaloNu PMT region (truth),7254.205,7254.205,1.000000
15,isDimuonn,7254.205,1.480,0.000204
12,Veto20 and Veto21 charge > 40 pC,1.480,1.470,0.993243
0,At least one long track,1.480,1.480,1.000000
4,Preshower Charge > 2.5 pC,1.470,1.445,0.982993
6,Timing Station Charge > 20 pC,1.470,1.470,1.000000
10,Track pz > 50 GeV,1.445,1.430,0.989619
1,Leading track has at >= 7 layers,1.430,1.430,1.000000
9,Track nDoF >= 9,1.430,1.420,0.993007
8,Track R at max radius < 95 mm,1.420,0.850,0.598592


In [14]:
print_cutflow("/home/ppd/bewilson/work/new_nuAnalysis/nuAnalysis/build/dimuon_selected/2024_CaloNu.bck.root")
print_cutflow("/home/ppd/bewilson/work/new_nuAnalysis/nuAnalysis/build/dimuon_selected/2024_CaloNu.bck.root", lumi_target=70)

,cut_name,all,passed,efficiency
5,Remove CaloNu PMT region (truth),1264895,1262287,0.997938
15,isDimuonn,1262287,159,0.000126
4,Preshower Charge > 2.5 pC,159,157,0.987421
0,At least one long track,159,159,1.000000
12,Veto20 and Veto21 charge > 40 pC,159,159,1.000000
6,Timing Station Charge > 20 pC,159,159,1.000000
10,Track pz > 50 GeV,157,156,0.993631
1,Leading track has at >= 7 layers,156,156,1.000000
9,Track nDoF >= 9,156,155,0.993590
8,Track R at max radius < 95 mm,155,87,0.561290


,cut_name,all,passed,efficiency
5,Remove CaloNu PMT region (truth),8854.265,8836.009,0.997938
15,isDimuonn,8836.009,1.113,0.000126
4,Preshower Charge > 2.5 pC,1.113,1.099,0.987421
0,At least one long track,1.113,1.113,1.000000
12,Veto20 and Veto21 charge > 40 pC,1.113,1.113,1.000000
6,Timing Station Charge > 20 pC,1.113,1.113,1.000000
10,Track pz > 50 GeV,1.099,1.092,0.993631
1,Leading track has at >= 7 layers,1.092,1.092,1.000000
9,Track nDoF >= 9,1.092,1.085,0.993590
8,Track R at max radius < 95 mm,1.085,0.609,0.561290


In [11]:
print_cutflow("/home/ppd/bewilson/work/new_nuAnalysis/nuAnalysis/build/dimuon_selected/2024_noCaloNu.bck.root", lumi_target=680)

,cut_name,all,passed,efficiency
5,Remove CaloNu PMT region (truth),98657.188,98657.188,1.000000
15,isDimuonn,98657.188,20.128,0.000204
12,Veto20 and Veto21 charge > 40 pC,20.128,19.992,0.993243
0,At least one long track,20.128,20.128,1.000000
4,Preshower Charge > 2.5 pC,19.992,19.652,0.982993
6,Timing Station Charge > 20 pC,19.992,19.992,1.000000
10,Track pz > 50 GeV,19.652,19.448,0.989619
1,Leading track has at >= 7 layers,19.448,19.448,1.000000
9,Track nDoF >= 9,19.448,19.312,0.993007
8,Track R at max radius < 95 mm,19.312,11.560,0.598592


In [16]:
print_cutflow("/home/ppd/bewilson/work/new_nuAnalysis/nuAnalysis/build/dimuon_selected/2000*.bck.root")

,cut_name,all,passed,efficiency
5,Remove CaloNu PMT region (truth),10635463,10635463,1.000000
15,isDimuonn,10635463,2278,0.000214
12,Veto20 and Veto21 charge > 40 pC,2278,2266,0.994732
0,At least one long track,2278,2278,1.000000
4,Preshower Charge > 2.5 pC,2266,2247,0.991615
6,Timing Station Charge > 20 pC,2266,2266,1.000000
10,Track pz > 50 GeV,2247,2218,0.987094
1,Leading track has at >= 7 layers,2218,2217,0.999549
9,Track nDoF >= 9,2217,2195,0.990077
2,Leading track has chi2/ndof < 15,2195,2192,0.998633


In [17]:
print_cutflow("/home/ppd/bewilson/work/new_nuAnalysis/nuAnalysis/build/2000*.bck.root")

,cut_name,all,passed,efficiency
0,At least one long track,10635463,64287,0.006045
4,Remove CaloNu PMT region (truth),10635463,10635463,1.000000
8,Veto20 and Veto21 charge > 40 pC,64287,58432,0.908924
5,Timing Station Charge > 20 pC,58432,58417,0.999743
3,Preshower Charge > 2.5 pC,58417,58046,0.993649
7,Track pz > 50 GeV,58046,48920,0.842780
1,Leading track has at >= 7 layers,48920,48892,0.999428
6,Track nDoF >= 9,48892,47870,0.979097
2,Leading track has chi2/ndof < 15,47870,47815,0.998851
9,VetoNu0 reduced charge < 30 pC,47815,46359,0.969549
